# Track 2 — Recommend Similar Songs

> **REFERENCE EXAMPLE — DO NOT SUBMIT UNCHANGED**
>
> This completed notebook demonstrates one reusable workflow. Run it to understand the decisions, then choose **a different dataset** for your final project and update the target or output, features, preprocessing, split, evaluation, and explanation to fit your data.


**Question:** Which tracks sound most similar to a selected song?

This reference uses a public Spotify Tracks dataset from Kaggle. You will build a nearest-neighbor recommender from audio features, compare it with random suggestions, and make the meaning of “similar” explicit.

### Finish line

- Inspect and clean track metadata and audio features.
- Scale features before measuring distance.
- Return five recommendations for a selected track.
- Compare neighbor distances with random recommendations.
- Change the feature set or weights and explain the effect.

Dataset: https://www.kaggle.com/datasets/yashdev01/spotify-tracks-dataset


In [ ]:
%pip -q install "kagglehub>=1.0.3"


In [ ]:
from pathlib import Path
import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
dataset_dir = Path(kagglehub.dataset_download("yashdev01/spotify-tracks-dataset"))

required = {"track_name", "artists", "danceability", "energy"}
csv_path = None
for candidate in dataset_dir.rglob("*.csv"):
    columns = set(pd.read_csv(candidate, nrows=2).columns)
    if required.issubset(columns):
        csv_path = candidate
        break

if csv_path is None:
    raise FileNotFoundError("Could not find the Spotify tracks CSV")

print("Using:", csv_path)


## 1. Inspect the data

One row represents one track. The dataset contains names, artists, genres, popularity, and numeric audio features.

We will not use popularity as a similarity feature: popular does not mean sonically similar.


In [ ]:
tracks = pd.read_csv(csv_path)
display(tracks.head())
print("shape:", tracks.shape)
display(tracks.isna().sum().sort_values(ascending=False).head(10).to_frame("missing"))
display(tracks.dtypes.to_frame("dtype"))


## 2. Choose what “similar” means

We use eight audio features that describe sound rather than identity. Distance models are sensitive to units, so every selected feature must be scaled before fitting nearest neighbors.


In [ ]:
audio_features = [
    "danceability", "energy", "loudness", "speechiness",
    "acousticness", "instrumentalness", "liveness", "valence",
]

clean = (
    tracks
    .dropna(subset=["track_name", "artists", *audio_features])
    .drop_duplicates(subset=["track_id"] if "track_id" in tracks.columns else ["track_name", "artists"])
    .reset_index(drop=True)
)

display(clean[audio_features].describe().T)
print("usable tracks:", len(clean))


In [ ]:
clean[audio_features].hist(figsize=(12, 8), bins=30)
plt.suptitle("Audio-feature distributions", y=1.02)
plt.tight_layout()
plt.show()


## 3. Fit the similarity pipeline

Standardization gives each feature comparable influence. Without it, loudness or another wide-range column could dominate the distance.


In [ ]:
scaler = StandardScaler()
feature_matrix = scaler.fit_transform(clean[audio_features])

neighbors = NearestNeighbors(n_neighbors=6, metric="euclidean")
neighbors.fit(feature_matrix)

def recommend(query, n=5):
    matches = clean[clean["track_name"].str.contains(query, case=False, regex=False)]
    if matches.empty:
        raise ValueError(f"No track name contains: {query!r}")

    # When several recordings share a title, start with the most popular match.
    if "popularity" in matches.columns:
        matches = matches.sort_values("popularity", ascending=False)

    selected_index = matches.index[0]
    distances, indices = neighbors.kneighbors(feature_matrix[[selected_index]], n_neighbors=n + 1)
    recommendation_indices = indices[0][1:]

    result_columns = ["track_name", "artists"]
    if "track_genre" in clean.columns:
        result_columns.append("track_genre")
    result_columns.extend(audio_features)

    result = clean.loc[recommendation_indices, result_columns].copy()
    result.insert(0, "distance", distances[0][1:])
    print("Selected:", clean.loc[selected_index, "track_name"], "—", clean.loc[selected_index, "artists"])
    return result


## 4. Produce a visible result

Start with a track name shown in the dataset. Replace the query with another title or part of a title.


In [ ]:
example_query = "bad guy"
display(recommend(example_query))


## 5. Compare with a baseline

A recommender has no single correct answer. As a simple quantitative check, compare nearest-neighbor distance with the distance to random tracks. A smaller distance only proves feature similarity—not that a listener will enjoy the result.


In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
query_indices = rng.choice(len(clean), size=100, replace=False)
nearest_distances, _ = neighbors.kneighbors(feature_matrix[query_indices], n_neighbors=2)

random_indices = rng.integers(0, len(clean), size=100)
random_distances = np.linalg.norm(feature_matrix[query_indices] - feature_matrix[random_indices], axis=1)

pd.DataFrame({
    "method": ["nearest neighbor", "random track"],
    "mean feature distance": [nearest_distances[:, 1].mean(), random_distances.mean()],
})


## 6. Inspect the feature explanation

The table below shows which scaled features differ most between the selected song and one recommendation. This is a simple explanation of the recommendation, not a causal claim.


In [ ]:
selected_index = 0
distances, indices = neighbors.kneighbors(feature_matrix[[selected_index]], n_neighbors=2)
recommended_index = indices[0][1]

explanation = pd.DataFrame({
    "feature": audio_features,
    "selected_scaled": feature_matrix[selected_index],
    "recommended_scaled": feature_matrix[recommended_index],
})
explanation["absolute_gap"] = (explanation["selected_scaled"] - explanation["recommended_scaled"]).abs()
display(explanation.sort_values("absolute_gap", ascending=False))


## 7. Try one change

- restrict recommendations to the same genre;
- give danceability or valence twice the weight;
- remove one feature and compare results;
- recommend movies, books, recipes, games, or courses using a different Kaggle dataset;
- ask three people whether the neighbors feel more relevant than random tracks.

Document what changed and one limitation of your evaluation.


In [ ]:
# Example: double the influence of danceability before fitting neighbors again.
feature_weights = np.ones(len(audio_features))
feature_weights[audio_features.index("danceability")] = 2.0
weighted_matrix = feature_matrix * feature_weights

# TODO: fit a second NearestNeighbors model on weighted_matrix and compare
# at least three recommendation lists with the original model.


## Required next step: choose your own dataset

The dataset in this notebook is only the completed example. For your final project, find another item dataset such as movies, books, recipes, games, products, or courses.

Before changing code, write down:

1. the new dataset source and license;
2. what one row or example represents;
3. the new target or desired output;
4. possible leakage or unavailable information;
5. the split strategy and why it matches real use;
6. the preprocessing steps that must change;
7. the baseline and main metric; and
8. one visible result you will produce with your data.

You may reuse the overall workflow and model as a starting point. You may not use the unchanged reference notebook as the final project.


## Completion checklist

- [ ] I can explain what one row represents.
- [ ] I chose features that match my definition of similarity.
- [ ] I removed duplicates and missing feature rows.
- [ ] I scaled features before measuring distance.
- [ ] I compared nearest neighbors with random recommendations.
- [ ] I changed one feature decision and inspected the result.
- [ ] I stated why feature similarity does not guarantee preference.
